##### This code extracts text from a word document and utilizes a pretrained model from Hugging Face to translate text fromenglish to spanish

In [3]:
pip install sentencepiece

     |████████████████████████████████| 1.1 MB 7.2 MB/s            
You should consider upgrading via the '/Users/reshea/opt/anaconda3/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [14]:
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.tokenize import sent_tokenize
from nltk.tokenize import LineTokenizer
from docx import Document
import docx
import re
import math
import torch
import os.path

In [27]:
def getText(doc):
    fullText = []
    # Reading in paragraphs
    for para in doc.paragraphs:
        fullText.append(para.text)
    # Reading in text from tables
    for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        fullText.append(paragraph.text)
    return '\n'.join(fullText)

In [28]:
def translate_paragraphs(paragraphs):
    lt = LineTokenizer()
    
    # To add more languages
    if torch.cuda.is_available():  
      dev = "cuda"
    else:  
      dev = "cpu" 
    device = torch.device(dev)

    tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ja-en")

    model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ja-en")
    model.to(device)
    
    # Translating sentences
    # Could choose batch size dynamically based on the length of the sentences.
    # Ideally no sentence/batch will be greater than 209 words
    batch_size = 8
    keys = []
    translated_paragraphs = []
    for paragraph in paragraphs:
        sentences = sent_tokenize(paragraph)
        batches = math.ceil(len(sentences) / batch_size)     
        translated = []
        for i in range(batches):
            # selecting the sentences to batch
            sent_batch = sentences[i*batch_size:(i+1)*batch_size]
            keys.extend(sent_batch)
            model_inputs = tokenizer(sent_batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            with torch.no_grad():
                translated_batch = model.generate(**model_inputs)
            translated += translated_batch
        translated = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
        translated_paragraphs += [" ".join(translated)]

    translated_text = "\n".join(translated_paragraphs)
    return translated_paragraphs, keys

In [29]:
def docx_replace(doc_obj, pairs, output_file_path):
    for key, value in pairs.items():
        for paragraph in doc_obj.paragraphs:
            replace_text_in_paragraph(paragraph, key, value)
        for table in doc_obj.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        replace_text_in_paragraph(paragraph, key, value)
    doc_obj.save(output_file_path)
    print("Done")

In [30]:
def replace_text_in_paragraph(paragraph, key, value):
    change_tracker = False
    if key in paragraph.text:
        inline = paragraph.runs
        for item in inline:
            if key in item.text:
                item.text = item.text.replace(key, value)
                change_tracker = True
        if change_tracker == False:
            paragraph.text = paragraph.text.replace(key, value)

In [31]:
def main(input_path, output_path):
    input_file = docx.Document(input_path)
    text = getText(input_file)
    lt = LineTokenizer()
    paragraphs = lt.tokenize(text)   
    translated_paragraphs, keys = translate_paragraphs(paragraphs)
    values = []
    for p in translated_paragraphs:
        values.extend(sent_tokenize(p))
    pairs = {keys[i]: values[i] for i in range(len(keys))}
    docx_replace(input_file, pairs, output_path)

In [32]:
main("Japanese_test_src.docx", "JT.docx")
# main("622_im_2.docx","investing_mixed_prime.docx")

Done


In [49]:
main("623_tt_1.docx", "table_test_prime.docx")

Done


In [61]:
sentences = ["This is a sentence", "This is another sentence", "These are sentences inside of a paragraph"]

['This is a sentence',
 'This is another sentence',
 'These are sentences inside of a paragraph']